In [1]:
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

C:\Users\harsh\anaconda3\envs\minor_project_env\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


In [3]:
MODEL_DIR = r"D:\MinorProject\MinorProject6thSem\models"

TEST_DIR = r"D:\Datasets\Phase1_224x224_RGB_JPEG50\test_unseen"

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # add SAME normalization used in training here
])

test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Testing on:", test_dataset.root)
print("Total images:", len(test_dataset))

Testing on: D:\Datasets\Phase1_224x224_RGB_JPEG50\test_unseen
Total images: 1500


In [5]:
class HybridModel(nn.Module):
    def __init__(self, num_classes=2):
        super(HybridModel, self).__init__()

        # EfficientNet branch
        eff_weights = models.EfficientNet_B0_Weights.DEFAULT
        self.effnet = models.efficientnet_b0(weights=eff_weights)
        eff_features = self.effnet.classifier[1].in_features
        self.effnet.classifier = nn.Identity()

        # ViT branch
        vit_weights = models.ViT_B_16_Weights.DEFAULT
        self.vit = models.vit_b_16(weights=vit_weights)
        vit_features = self.vit.heads.head.in_features
        self.vit.heads = nn.Identity()

        # Fusion
        self.classifier = nn.Sequential(
            nn.Linear(eff_features + vit_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        eff_feat = self.effnet(x)
        vit_feat = self.vit(x)
        fused = torch.cat([eff_feat, vit_feat], dim=1)
        return self.classifier(fused)

In [6]:
criterion = nn.CrossEntropyLoss()

def evaluate_full(model, loader, device):

    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    running_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    total_loss = running_loss / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)

    print("\nLoss:", round(total_loss,4))
    print("Accuracy:", round(acc,4))
    print("Precision:", round(prec,4))
    print("Recall:", round(rec,4))
    print("F1:", round(f1,4))
    print("AUC:", round(auc,4))
    print("Confusion Matrix:\n", cm)

    return {
        "loss": total_loss,
        "acc": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc
    }

In [7]:
eff = models.efficientnet_b0(weights=None)
eff.classifier[1] = nn.Linear(eff.classifier[1].in_features, 2)

eff.load_state_dict(
    torch.load(
        MODEL_DIR + r"\best_efficientnetb0_rgb_phase1.pth",
        map_location=DEVICE
    )
)

eff = eff.to(DEVICE)

print("\n===== EfficientNetB0 =====")
results_eff = evaluate_full(eff, test_loader, DEVICE)

C:\Users\harsh\AppData\Local\Temp\ipykernel_19956\1066174610.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(



===== EfficientNetB0 =====

Loss: 0.1312
Accuracy: 0.968
Precision: 0.938
Recall: 0.968
F1: 0.9528
AUC: 0.9949
Confusion Matrix:
 [[968  32]
 [ 16 484]]


In [8]:
vit = models.vit_b_16(weights=None)
vit.heads.head = nn.Linear(vit.heads.head.in_features, 2)

vit.load_state_dict(
    torch.load(
        MODEL_DIR + r"\best_vitb16_rgb_phase1.pth",
        map_location=DEVICE
    )
)

vit = vit.to(DEVICE)

print("\n===== ViT-B16 =====")
results_vit = evaluate_full(vit, test_loader, DEVICE)

C:\Users\harsh\AppData\Local\Temp\ipykernel_19956\2681315091.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(



===== ViT-B16 =====

Loss: 1.2161
Accuracy: 0.6993
Precision: 0.5258
Recall: 1.0
F1: 0.6892
AUC: 0.9991
Confusion Matrix:
 [[549 451]
 [  0 500]]


In [9]:
hybrid = HybridModel(num_classes=2).to(DEVICE)

hybrid.load_state_dict(
    torch.load(
        MODEL_DIR + r"\best_hybrid_rgb_phase1.pth",
        map_location=DEVICE
    )
)

print("\n===== Hybrid Model =====")
results_hybrid = evaluate_full(hybrid, test_loader, DEVICE)

C:\Users\harsh\AppData\Local\Temp\ipykernel_19956\3007844076.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(



===== Hybrid Model =====

Loss: 0.3786
Accuracy: 0.87
Precision: 0.7194
Recall: 1.0
F1: 0.8368
AUC: 0.996
Confusion Matrix:
 [[805 195]
 [  0 500]]
